# Introduction to Natural Language Processing and Generative Language Modelling

This chapter motivates some basic concepts which we need to understand how neural network can produce text. Before we discuss state-of-the-art techniques like GPT, let's have a look at some older techniques.

## Classic Natural Language Processing (NLP)

In [ ]:
import nltk  # the natural language toolkit is THE library for classic NLP

# we need to download some packages for nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')
nltk.download('vader_lexicon')

Many classic NLP techniques wanted to make machines **really** understand language, down to the grammatical level. With the hypothesis being that "if machines can truly understand language, then they can also use it". This is quite different from the approach we are taking today (as we will see shortly).

### Part of speech tagging

In [ ]:
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize

sentence = "Samantha said: Let's go to London this weekend!"
pos_tag(word_tokenize(sentence), tagset='universal', lang="eng") 

_... not terribly exciting ..._

### Sentiment analysis

Classic NLP also included some classification and summary tasks like sentiment analysis. The standard procedure would have been to obtain a dataset suitable for the task you want to solve and train a model specifically for that task.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()

In [ ]:
sid.polarity_scores("This movie is great, really funny")

In [ ]:
sid.polarity_scores("This is a terrible movie")

## First steps in text generation using classic NLP

We are now going to demonstrate next-word prediction using some classic NLP techniques. 

Nobody would have really used the techniques below to generate text, because the results are just not that good.

But the general idea of "let's predict the next word given some context words" is still what drives modern LLMs today.

### Example dataset

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
text_path = Path("../.assets/data/grundgesetz/Grundgesetz_für_die_Bundesrepublik_Deutschland_(1949).txt")
text = text_path.read_text()

# split into individual words
# usually we would do some additional text cleaning, stemming, etc; let's skip that for today
text = text.split()

print(text[0:50])

### Unigrams

In [ ]:
# let's count how often each word appears in the data
unigrams = pd.Series(text, name="word").value_counts()
unigrams = unigrams.to_frame()
unigrams

In [ ]:
# if we divide the counts by the total number of words, we get the frequency of each word
unigrams["frequency"] = unigrams["count"] / unigrams["count"].sum()
unigrams

#### Randomly sample from probabilities

We want to generate text that follows the _distribution_ of our original text. This means, for example, that the frequency of words in the generated text should be similar to the frequency of words in the original text.

Let's have a look at how we can make that happen:

* idea: each word "takes up space" proportional to its frequency in the training data
* generate a random number and "look up" to which word it belongs.
* ⇒ More frequent words will be sampled more often.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/sampling.svg" style="max-width: 700px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Sampling strategies for text generation
  </div>
</div>

In [ ]:
# calculate the cumulative frequency (numbers in grey in the image above)
unigrams["cumulative frequency"] = unigrams["frequency"].cumsum()
unigrams

##### Sampling function

In [ ]:
# let's implement our sampling function as lined out above
def sample(cumulative_frequencies: pd.Series):
    # get a random number
    rand = random.random()
    
    # get all words whose cumulative frequency is above that random number
    subset = cumulative_frequencies[cumulative_frequencies > rand]
    
    # we want the first of those words
    return subset.index[0]

##### Sample iteratively

In [ ]:
result = []

# iterative sampling
for i in range(40):
    sampled = sample(unigrams["cumulative frequency"])
    result.append(sampled)
    
" ".join(result)

⇒ _this doesn't make much sense. Our language model is too simple, because it only looks at individual words_

### Bigrams: introducing the idea of context

_Let's be a bit more clever and count how often each word is followed by other words_

In [ ]:
bigrams = pd.DataFrame({
    "word": text, 
    "next word": text[1:] + [None]  # we shift text by 1 to get the next word
})
bigrams

In [ ]:
# now we can count how often each word-combination occurs 
bigrams = bigrams.groupby(["word", "next word"]).size().to_frame("count")
bigrams = bigrams.sort_values("count", ascending=False)

# these are the most common bigrams in the whole text
bigrams

In [ ]:
# these are all the words that follow "der" and how often each of them follows "der"
bigrams.loc["der"]

In [ ]:
# these are the same calculations we did above for the unigrams
def calculate_frequencies(group):
    group["frequency"] = group["count"] / group["count"].sum()
    group["cumulative frequency"] = group["frequency"].cumsum()
    return group
    

# but now we are doing them for each first word in the bigram
bigrams = bigrams.groupby("word", group_keys=False).apply(calculate_frequencies)

# again, these are the words that follow "der" with their counts and frequencies
bigrams.loc["der"]

##### Sample iteratively

In [ ]:
result = ["der"]  # start context
    
for i in range(40):
    # context = most recent word
    context = result[-1] 
    
    # sample from words that typically follow our context word
    subset = bigrams.loc[context]
    sampled = sample(subset["cumulative frequency"])  
        
    result.append(sampled)
        
print(" ".join(result))

⇒ _Still not very good, but the text looks a little more sensible._ <br/> 

_A larger, more diverse training dataset and using trigrams (i.e. context = 2 words) or more would help improve things further._

## Recurrent neural networks

Recurrent neural networks (RNNs) had been around since the 1970s, but started to get really big in the 2010s as part of the deep learning boom. 

RNNs were/are used for all kinds of NLP tasks: translation, summarization, text generation, sentiment analysis, etc.

The basic idea of RNNs is: instead of doing our simple counting logic, we use a neural network to identify relations between words.

### How do RNNs work?

On the right hand side of the diagram below you can see:

1. An input token/word $X_{t-1}$ (green) is passed into the neural network (blue)
2. The neural network generates an output token/word $O_{t-1}$ (red)
3. The neural network keeps some state $h_{t-1}$, which encapsulates some "knowledge" about the most recently seen words
4. The next input $X_{t}$ is passed into the same network.
5. The network updates it's state and produces the next output $O_t$.
6. goto4

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/rnn_unfold.svg" style="max-width: 700px; max-height: 400px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Recurrent neural network unfolded — by fdeloche, <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>, via <a href="https://commons.wikimedia.org/wiki/File:Recurrent_neural_network_unfold.svg">Wikimedia Commons</a>
  </div>
</div>

### Why did RNNs fall out of fashion?

When the "transformer" model architecture was introduced in 2017, RNNs started quickly falling out of fashion.

The main reasons for that were:
* RNNs are slow to train, because one always needs to feed in tokens step by step
  * In contrast, with the transformer architecture we can train multiple timesteps at the same time
* RNNs are complex, which makes training complicated and unstable
  * In contrast, the transformer architecture does not have any recursive layers, which makes it easier to train
* The hidden state in RNNs summarise the context, which for many tasks (e.g. translation) is not enough
  * In contrast, the transformer architecture has access to the full context

## The transformer

The transformer is a deep learning architecture proposed in 2017 that has taken the world by storm.

Why is it called transformer? Nobody seems to be sure, I like to think it is because it was originally proposed for machine translation, i.e. transforming one language into the other. But probably somebody just thought it was a fun name.

### Intuition

If we look back at the RNNs, we can see, that there the input is fed into the model word-by-word.

In contrast, the transformer get's based the whole input context at the same time. This means it has more information available for making its prediction.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_intuition.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Transformer attention intuition
  </div>
</div>

The input context can become quite large, in ChatGPT, the context can be up to 3000 token (about 2000 words) long. 

In order to help the model focus on those part of the context that are most important for predicting the next word, the transformer includes the so-called _attention_ layers.

For the example sentence, the model might have calculated that it is most important to focus on the subject of the sentence ("She") and what the subject is doing ("talking"):

In [ ]:
attention = {
    "She": 0.3,
    "was": 0.1,
    "talking": 0.4,
    "to": 0.2
}   

Using the attention information, the model can now do a weighted aggregation of the input context and use that summarised information to predict the next word.

Please note that the attention mechanism is also trained, i.e. during training the model learns to which pieces of the input it should give attention to in different situations.

### Full transformer architecture

We will not go into detail in this notebook, but we do have another notebook with a deep dive into the transformer architecture and the attention mechanism, if you are interested. 

One thing that is notable with this architecture is that there is no recurrent connection, i.e. no arrows that come out of some layer and feed back into that same layer.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_architecture_cross_attention.png" style="max-width: 500px; max-height: 700px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Transformer architecture with cross-attention — Figure 1 from Vaswani et al., <a href="https://arxiv.org/abs/1706.03762">"Attention Is All You Need"</a> (2017)
  </div>
</div>

### Emergent Capabilities: Is scaling all you need?

We have seen several advantages of the transformer architecture compared to RNNs:

* Transformer takes all context tokens at the same time ⇒ faster to train.
* Transformer is not recursive ⇒ easier to train
* Transformer has access to full context ⇒ more capable model

Thanks to these innovations, researches were able to train much much larger language models than before. And found that larger models, despite note being fine-tuned, are able to beat specialised models on a number of tasks.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_scaling1.png" style="max-width: 700px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Accuracy on the TriviaQA benchmark grows with model size — Figure 3.3 from Brown et al., <a href="https://arxiv.org/abs/2005.14165">"Language Models are Few-Shot Learners"</a> (2020)
  </div>
</div>

### Zero-shot, One-shot, Few-Shot - the prompt innovation

We have learned earlier, that with classic NLP it was common to train a dedicated model from scratch for each task.

With RNNs, this situation improved, and there already was some reusability.

But only with transformer-based large language models do we now have foundation models that we can apply to many different tasks.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/transformers_fewshot.png" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Zero-shot, one-shot and few-shot prompting vs. fine-tuning — Figure 2.1 from Brown et al., <a href="https://arxiv.org/abs/2005.14165">"Language Models are Few-Shot Learners"</a> (2020)
  </div>
</div>

## Next word prediction as the backbone of NLP

We saw earlier that in classic NLP there was a focus on teaching models to understand our language, down to the grammar level.

This thought is also **completely** out of fashion in the NLP landscape today. The **only** thing we teach our models to do is to generate the next word given some context. 

And somehow, capabilities that have nothing to do with next-word prediction emerge. How? Why? (_starting point for discussion_)


---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_